# 02 — Обзор выбросов и аномальных строк

Этот notebook отдельно от основного EDA отмечает статистические кандидаты по IQR и robust Z-score (MAD). Он **не изменяет, не удаляет и не исправляет данные**.

In [ ]:
from pathlib import Path
import importlib
import sys

from IPython.display import display
import matplotlib.pyplot as plt

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").exists() and (candidate / "src").exists()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project import (
    DataCatalog,
    MarkdownDocument,
    analyze_numeric_anomalies,
    plot_anomaly_summary,
    plot_feature_anomalies,
    validate_feature_groups,
)
from ml_project.docsync import dataframe_to_markdown
import ml_project.config as project_config

IQR_MULTIPLIER = 1.5
ROBUST_Z_THRESHOLD = 3.5
TOP_ROWS = 30

print(f"Корень проекта: {PROJECT_ROOT}")

## 1. Загрузить те же EDA-данные

Границы всегда рассчитываются только по train и без изменений применяются к inference/test.

In [ ]:
project_config = importlib.reload(project_config)
catalog = DataCatalog(
    PROJECT_ROOT,
    project_config.RAW_DIR,
    project_config.DATASETS,
)
catalog.validate()
datasets = catalog.load_all()

EDA_ROLES = {"train", "inference"}
eda_datasets = {
    name: frame
    for name, frame in datasets.items()
    if project_config.DATASETS[name].get("role") in EDA_ROLES
}
train = eda_datasets[project_config.TRAIN_DATASET]
feature_groups = validate_feature_groups(
    train,
    project_config.FEATURE_GROUPS,
    target=project_config.TARGET,
)
numeric_features = [
    *feature_groups["numeric"],
    *feature_groups["count"],
]
print("Датасеты:", ", ".join(eda_datasets))
print("Числовые + счётные:", ", ".join(numeric_features) or "—")

## 2. Рассчитать обзор

IQR и MAD — сигналы для просмотра, а не правило удаления строки. Если IQR или MAD равен нулю, соответствующий метод для признака честно отключается.

In [ ]:
anomaly_report = analyze_numeric_anomalies(
    eda_datasets,
    reference_dataset=project_config.TRAIN_DATASET,
    columns=numeric_features,
    iqr_multiplier=IQR_MULTIPLIER,
    robust_z_threshold=ROBUST_Z_THRESHOLD,
)

display(anomaly_report.feature_bounds.round(4))
display(
    anomaly_report.dataset_summary.style.format(
        {"outlier_share": "{:.2%}"}
    )
)
axis = plot_anomaly_summary(anomaly_report)
axis.figure.tight_layout()
plt.show()

## 3. Посмотреть отмеченные значения

Показываются только признаки, где хотя бы один датасет получил IQR/MAD-флаг. Красный крест — отмеченное значение, пунктир — границы train-IQR.

In [ ]:
flagged_features = (
    anomaly_report.dataset_summary.loc[
        anomaly_report.dataset_summary["any_outliers"].gt(0),
        "feature",
    ]
    .drop_duplicates()
    .tolist()
)
if not flagged_features:
    print("IQR/MAD-кандидаты не найдены.")
else:
    for feature in flagged_features:
        for dataset_name, frame in eda_datasets.items():
            summary_row = anomaly_report.dataset_summary[
                anomaly_report.dataset_summary["dataset"].eq(dataset_name)
                & anomaly_report.dataset_summary["feature"].eq(feature)
            ].iloc[0]
            if summary_row["any_outliers"] == 0:
                continue
            axis = plot_feature_anomalies(
                frame,
                anomaly_report,
                dataset=dataset_name,
                feature=feature,
            )
            axis.figure.tight_layout()
            plt.show()

## 4. Разобрать конкретные строки

`feature_count` показывает, в скольких признаках строка необычна. Сначала смотрим строки с несколькими флагами, затем самые сильные одиночные отклонения.

In [ ]:
display(anomaly_report.flagged_values.head(100).round(4))
display(anomaly_report.row_summary.head(TOP_ROWS).round(4))

context_columns = list(dict.fromkeys(
    [project_config.KEY, project_config.TARGET, *numeric_features]
))
for dataset_name, part in anomaly_report.row_summary.groupby("dataset", sort=False):
    top_rows = part.head(TOP_ROWS)
    source = eda_datasets[dataset_name]
    available_columns = [column for column in context_columns if column in source]
    print(dataset_name.upper())
    display(source.iloc[top_rows["row_position"].astype(int)][available_columns])

## 5. Синхронизировать только фактическую сводку

В `docs/02_eda.md` попадут метод и количества. Объяснение и решение остаются ручными.

In [ ]:
docs_summary = anomaly_report.dataset_summary[
    [
        "dataset",
        "feature",
        "non_missing",
        "iqr_outliers",
        "mad_outliers",
        "any_outliers",
        "outlier_share",
        "outside_reference_range",
    ]
].copy()
docs_summary["outlier_share"] = docs_summary["outlier_share"].map(
    lambda value: f"{value:.2%}"
)
docs_summary = docs_summary.rename(
    columns={
        "dataset": "Датасет",
        "feature": "Признак",
        "non_missing": "Непустых",
        "iqr_outliers": "IQR",
        "mad_outliers": "MAD",
        "any_outliers": "Любой метод",
        "outlier_share": "Доля",
        "outside_reference_range": "Вне диапазона train",
    }
)
anomaly_block = (
    "Автоматический обзор: [[notebooks/02_eda_anomalies.ipynb|02_eda_anomalies.ipynb]]. "
    f"Границы рассчитаны по `{project_config.TRAIN_DATASET}`: "
    f"IQR × {IQR_MULTIPLIER:g}, robust Z-score через MAD ≥ {ROBUST_Z_THRESHOLD:g}. "
    "Флаги требуют ручной интерпретации и не удаляют строки.\n\n"
    + dataframe_to_markdown(docs_summary)
)
updated = MarkdownDocument(PROJECT_ROOT / "docs/02_eda.md").update_blocks(
    {"eda-anomalies": anomaly_block}
)
print("Обновлены блоки:", ", ".join(updated))

## 6. Зафиксировать смысл

Статистический флаг не равен ошибке данных. Просмотрите пассажира в контексте и заполните ручную таблицу раздела «Пропуски, выбросы и аномалии». Если наблюдение ведёт к модельной идее, сохраните его через `02_eda_hypotheses.ipynb` и проверьте отдельным экспериментом.